In [1]:
import os, glob
import math
import pickle
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader

from pl_regressor import RNARegressor
from fsp import generate
from fsp_utils import (
    CELLTYPE_CODES_UTR3,
    CELLTYPE_CODES_UTR5,
    predictor,
)

## Config

Switch `UTR_TYPE` between `"utr3"` and `"utr5"` to run the corresponding experiment.

In [3]:
# ---- experiment config ----
UTR_TYPE = "utr5"  # "utr3" or "utr5"
DEVICE = "cuda:0"
SEED = 42

# model
MODEL_PATH = glob.glob(f"../../predictor/regression_multiple/saved_models/model-{UTR_TYPE}-deltas*.ckpt")[0]

# generation options
SEQ_LEN = 50 if UTR_TYPE == 'utr5' else 240
BATCH_SIZE = 128
N_SAMPLES_TRAIN = 8
N_STEPS = 200
N_PROPOSALS = 2000  # Change to generate more sequences
LOSS_TYPE = "malinois"

# which cell types to generate for
CELLTYPE_CODES = CELLTYPE_CODES_UTR5 if UTR_TYPE == "utr5" else CELLTYPE_CODES_UTR3
CELL_TYPE_COLS = list(CELLTYPE_CODES.keys())

np.random.seed(SEED)
torch.manual_seed(SEED) 

## Load model

In [4]:
model_predict = RNARegressor.load_from_checkpoint(MODEL_PATH, map_location="cpu").model.to(DEVICE)
model_generate = RNARegressor.load_from_checkpoint(MODEL_PATH, map_location="cpu").model.to(DEVICE)
model_predict.eval()
model_generate.eval()
print('Loaded')

Loaded


## FastSeqProp generation

In [ ]:
seq_dict, _ = generate(
    model_generate,
    seq_len=SEQ_LEN,
    device=DEVICE,
    n_samples_train=N_SAMPLES_TRAIN,
    n_steps=N_STEPS,
    n_proposals=N_PROPOSALS,
    utr_type=UTR_TYPE,
    CELLTYPE_CODES_FORGEN=CELLTYPE_CODES,
    batch_size=BATCH_SIZE,
    loss_type=LOSS_TYPE,
    create_plot=False,
)

Steps:  38%|███▊      | 76/200 [00:57<01:33,  1.33it/s, Loss=-.511, LR=0.363]

In [ ]:
df_list = []

for cell_seq in seq_dict:
    pred_df = predictor(
        torch.concat(seq_dict[cell_seq]),
        cell_seq,
        model_predict,
        seq_len=SEQ_LEN,
        device=DEVICE,
        utr_type=UTR_TYPE,
    )
    df_list.append(pred_df)

df_cell_pred = pd.concat(df_list, ignore_index=True)
df_cell_pred.to_csv(f"FSP_{UTR_TYPE}_generated.csv", index=False)
df_cell_pred.head()